# Explore Model

Interactive notebook for loading a model, running inference, and visualizing training results.

In [ ]:
import sys
sys.path.insert(0, '..')  # project root

from app.core.gpu_utils import get_device_info
import json

info = get_device_info()
print(json.dumps(info, indent=2))

## Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "microsoft/phi-2"  # change to your model
ADAPTER_PATH = None           # set to adapter path if using LoRA

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

if ADAPTER_PATH:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)

model.eval()
print(f"Model loaded: {MODEL_ID}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## Run Inference

In [ ]:
prompt = "<|user|>\nWhat is LoRA fine-tuning?\n<|assistant|>\n"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

## Visualize Training Loss

In [ ]:
import json
from pathlib import Path

# Adjust path to your adapter output directory
state_file = Path("../models/adapters/phi2-lora/trainer_state.json")

if state_file.exists():
    state = json.loads(state_file.read_text())
    history = [e for e in state["log_history"] if "loss" in e]
    steps = [e["step"] for e in history]
    losses = [e["loss"] for e in history]

    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(10, 4))
        plt.plot(steps, losses, linewidth=2)
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("Training Loss")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except ImportError:
        for step, loss in zip(steps, losses):
            print(f"Step {step:>6}: {loss:.4f}")
else:
    print(f"No trainer_state.json found at {state_file}")
    print("Run a training job first with: python training/train.py --config configs/phi_lora.yaml")